In [ ]:
import numpy as np # math library, simulation of random variables
import matplotlib.pyplot as plt
import seaborn as sns # more fancy stats visualization
from scipy import stats # stats library
from scipy import optimize # optimization library 
import statsmodels.api as sm # statistical modeling library

In this notebook, we run simulation about a hypothetical clinical trial scenario, in which we are comparing two drugs, with respective (unknown) probability of success $p_1$ and $p_2$. We are interested in testing $\mathcal{H}_0 : (p_1 = p_2)$ against $\mathcal{H}_1 : (p_1 \neq p_2)$ based on the observation of the efficacy of the treatments for $n_1$ patients who received treatment $1$, $X=(X_1,\dots,X_{n_1})$ and $n_2$ patients who received treatment $2$, $Y=(Y_1,\dots,Y_{n_2})$. 

All the observations are assumed to be independent with $X_i \sim \mathcal{B}(p_1)$ and $Y_{j} \sim \mathcal{B}(p_2)$. 

We will explore numerically the properties of two tests seen in class: the Wald test and the (Generalized) Likelihood ratio test. 

## Generating Bernoulli data

In [ ]:
p1 = 0.2 # probability of success of treatment 1
p2 = 0.6 # probability of success of treatment 2

n1 = 100 # number of patients in group 1
n2 = 75 # number of patients in group 2

rng = np.random.RandomState(42)

X = (rng.random(n1)<p1).astype(int)
print("average efficacy of treatment 1 is", np.mean(X))

Y = (rng.random(n2)<p2).astype(int)
print("average efficacy of treatment 2 is", np.mean(Y))

# Wald test 

We recall the principle of the Wald test. If our test is about a parameter being equal to a certain value (here $p_2-p_1 = 0$), propose an estimator of this parameter, derive its asymptotic distribution (with a data-dependent variance), and deduce a test statistic that is asymptotically normal under $H_0$. 

In our case, the (MLE) estimator of the treatment effect is $\hat{p}_{2,n_2} - \hat{p}_{1,n_1}$ where $\hat{p}_{1,n_1} = \frac{1}{n_1}\sum_{i=1}^{n_1} X_i$ is the average efficacy for $n_1$ patients who received treatment 1, and $\hat{p}_{2,n_2} = \frac{1}{n_2}\sum_{i=1}^{n_2} Y_i$ is the average efficacy for $n_2$ patients who received treatment 2. 

In class, we proved that when $n_1=n_2=n$, for any $p_1,p_2$, 
$$\frac{\hat{p}_{2,n} - \hat{p}_{1,n} - (p_2 - p_1)}{\sqrt{\frac{\hat{p}_{1,n}(1-\hat{p}_{1,n})}{n} + \frac{ \hat{p}_{2,n}(1- \hat{p}_{2,n})}{n}}}$$ 
converges in distribution to a $\mathcal{N}(0,1)$ random variable. In particular, under $\mathcal{H}_0$ (that is, when $p_1 = p_2$), the test statistic 
$$W_n = \frac{\hat{p}_{2,n} - \hat{p}_{1,n}}{\sqrt{\frac{\hat{p}_{1,n}(1-\hat{p}_{1,n})}{n} + \frac{ \hat{p}_{2,n}(1- \hat{p}_{2,n})}{n}}}$$
is asymptotically a standard Gaussian. The Wald test with threshold $t$ is given by 
$$D(X) = \mathbb{1}(|W_n| > t).$$

The Wald test may also be used for two populations of different size, by defining 
$$W_n = \frac{\hat{p}_{2,n_2} - \hat{p}_{1,n_1}}{\sqrt{\frac{\hat{p}_{1,n_1}(1-\hat{p}_{1,n_1})}{n_1} + \frac{ \hat{p}_{2,n_2}(1- \hat{p}_{2,n_2})}{n_2}}}$$
which is also asymptotically standard Gaussian when $n_1$ and $n_2$ tend to infinity. 


**Complete the code below with the threshold that guarantees an asymptotic level of significance equal to alpha.**


In [ ]:
def Wald_test(X,Y,alpha=0.05):
    n1 = len(X)
    n2 = len(Y)
    p1_hat = np.mean(X)
    p2_hat = np.mean(Y)
    var1 = p1_hat*(1-p1_hat)
    var2 = p2_hat*(1-p2_hat)
    if (var1 ==0)&(var2==0):
        # avoid division by zero
        return 0
    else:
        test_statistic = (p2_hat - p1_hat)/np.sqrt(var1/n1 + var2/n2) 
        threshold = ## TO COMPLETE
        decision = 0
        if (np.abs(test_statistic)>threshold):
            decision = 1
        return decision 

## Running the test

The cell below is running the Wald for given values of $p_1$, $p_2$ and the number of patients $n$ on simulated Bernoulli data. 

**You can try different values of the parameters and different seeds (corresponding to different observations, that can be viewed as observations for different groups of patients), and see when the test is rejecting or keeping H0**

In [ ]:
# success probabilities
p1 = 0.8
p2 = 0.8

# number of patients
n1 = 50 
n2 = 75

rng = np.random.RandomState(42)

X = (rng.random(n1)<p1).astype(int)
print("average efficacy of treatment 1 is", np.mean(X))
Y = (rng.random(n2)<p2).astype(int)
print("average efficacy of treatment 2 is", np.mean(Y))

print("decision of the Wald test:", Wald_test(X,Y))

## Visualizing the distribution of the test statistic

In the sequel, we will always assume $n_1=n_2=n$ to simplify the exposition (but the test we will see can all be defined for unbalanced sample sizes as well). 

A good test should use a test statistic whose *distribution* is significantly different from the null to an alternative, so as to make it very unlikely to reject when $\mathcal{H}_0$ is true (small type one error) and very likely to reject when $\mathcal{H}_1$ is true (large power / small type II error).  

Under $\mathcal{H}_1$, that is when $p_1 \neq p_2$, the statistic $T_n$ satisfies 
$$T_n = Z_n + \frac{p_2 - p_1}{\sqrt{\frac{\hat{p}_{1,n}(1-\hat{p}_{1,n})}{n} + \frac{ \hat{p}_{2,n}(1- \hat{p}_{2,n})}{n}}}$$
where $Z_n$ converges in distribution to a $\mathcal{N}(0,1)$, so we write
$$T_n \simeq \mathcal{N}\left( \frac{(p_2 - p_1)\sqrt{n}}{\sqrt{p_1(1-p_1) + p_2(1-p_2)}}, 1\right)$$
whereas under $\mathcal{H}_0$, 
$$T_n \simeq \mathcal{N}\left(0, 1\right)$$

The code below visualizes the density of the asymptotic distribution of $T_n$ when $p_1=p_2$ and for the point $p_1=0.8,p_2=0.9$ in the alternative as well as the boundaries of the rejection region (which is outside the two vertical lines).  

**Where do we read the power (at this specific alternative) on the plot? You can play with the parameters and see how the power evolves with the sample size, the gap between p1 and p2 and the type I error alpha**

In [ ]:
p1=0.8
p2=0.9

n = 200
alpha = 0.05

mean_h1 = np.sqrt(n)*(p2-p1)/(np.sqrt(p1*(1-p1)+p2*(1-p2)))

law1=stats.norm
law2=stats.norm(mean_h1,1)

thres = law1.ppf(1-alpha/2)

xplot = np.linspace(-np.abs(mean_h1)-3,np.abs(mean_h1)+3,num=200)
plt.plot(xplot,law1.pdf(xplot),label="$T_n$ under H0")
plt.plot(xplot,law2.pdf(xplot),label="$T_n$ under H1")
plt.axvline(thres, 0, 1, c='black', linewidth = 1)
plt.axvline(-thres, 0, 1, c='black', linewidth = 1)
plt.legend()
plt.show()

# power computation 
#beta = ## TO COMPLETE
#print("(asymptotic) power at this alternative is",beta)

**Based on this approximation, how many samples are needed to get 80% power at the alternative $p_1=0.8, p_2=0.9$ ?**

## Approximation of the test characteristics for a finite sample size 

The above reasonning is purely asymptotic and uses some approximations, but we can use some simulation to actually *estimate* the type I error and power, for a finite sample size. 

Indeed, for $p_1=p_2=p$, 
$$\alpha((p,p)) = \mathbb{P}_{(p,p)} (D_n(X,Y) = 1) = \mathbb{E}_{(p,p)}[D_n(X,Y)]$$
which can be estimated using an empirical average. 

Given the simulation of $M$ experiments, giving the observations $(X^{m},Y^{m})$ for $m=1,\dots,M$, we can propose the estimate 
$$\widehat{a}_M = \frac{1}{M}\sum_{m=1}^{M} D_n(X^{m},Y^{m})\;.$$
That is, we run $M$ times the test on $M$ different dataset of patients, and report the average number of rejections. 

**Try different seed to get of sense of the variability in estimating the type I error.**

In [ ]:
p=0.8 # common value of p1 and p2 for an element in H0

alpha_0 = 0.05 # desired type I error 

rng=np.random.RandomState(75)

M = 1000 # number of simulations 
n_values = np.array([10,20,30,50,100,200,300,500,800,1000]) # number of patients for which we want to estimate the type I error  
n_max =np.max(n_values)
n_nb = len(n_values)

X_table = (rng.random(size=(M,n_max))<p).astype(int)
Y_table = (rng.random(size=(M,n_max))<p).astype(int)

rejects = np.zeros(n_nb)

for i in range(n_nb):
    n = n_values[i] 
    for m in range(M):
        X_small = X_table[m,range(n)]
        Y_small = Y_table[m,range(n)]
        decision=Wald_test(X_small,Y_small,alpha_0)
        rejects[i]=rejects[i]+decision


plt.plot(n_values,rejects/M)

plt.axhline(alpha_0,0, 1, c='black', linewidth = 1)
plt.title("empirical type I error at p1=p2={} as a function of the sample size".format(p))
plt.show()


**For small values of $n$, is the asymptotic calibration valid?** 

To answer this question, you might want to display confidence intervals on the values of alpha to account for the possible variability.

Check out [this function](https://www.statsmodels.org/dev/generated/statsmodels.stats.proportion.proportion_confint.html).

In [ ]:
small=(n_values < 100)
n_small=n_values[small]
rejects_small=rejects[small]

ci_low, ci_upp=sm.stats.proportion_confint(count=rejects_small,nobs=M*np.ones(len(rejects_small)),alpha=0.05,method='beta')

plt.plot(n_small,ci_upp,label="upper bound on alpha")
plt.plot(n_small,rejects_small/M)
plt.plot(n_small,ci_low,label="lower bound on alpha")


plt.axhline(alpha_0,0, 1, c='black', linewidth = 1)

plt.legend()


For $n\leq 25$ the type I error is confidently above $\alpha=0.05$.

We can do similar simulations for the power, estimated as 
$$\frac{1}{M}\sum_{m=1}^{M} D_n(X^{m},Y^{m})\;.$$
(fraction of rejections) but when the data is simulated under an alternative. 

**How many samples seem necessary to get a power 0.8 under the alternative p1=0.8, p2=0.9?**

# Likelihood-Ratio Test

For Bernoulli data, the log-likelihood ratio statistic associated to the test 
$\mathcal{H}_0 : (p_1 = p_2)$ against $\mathcal{H}_1 : (p_1\neq p_2)$ is 

$$\log \widetilde{\Lambda}(X,Y) = \log\frac{\sup_{p_1 \in [0,1]} p_1^{S_1}(1-p_1)^{n-S_1}\sup_{p_2 \in [0,1]} p_2^{S_2}(1-p_2)^{n-S_2}}{\sup_{p \in [0,1]} p^{S_1}(1-p)^{n-S_1}p^{S_2}(1-p)^{n-S_2}} $$
where $S_1$ is the number of successes of treatment 1 among the $n$ patients who received it and $S_2$ is the number of successes of treatment 2 among the $n$ patients who received it.

Introducing the estimates $\hat{p}_{1,n}$, $\hat{p}_{2,n}$ and $\widehat{p}_{n} = \frac{1}{2n}\sum_{i=1}^{n}(X_i+Y_i)$ (the average efficacy across all patients), we have (dropping the index $n$ for readability)

\begin{align}
\log \widetilde{\Lambda}(X,Y) & = \log\frac{\hat{p}_1^{S_1}(1-\hat{p}_1)^{n-S_1}\hat{p}_2^{S_2}(1-\hat{p}_2)^{n-S_2}}{\hat{p}^{S_1}(1-\hat{p})^{n-S_1}\hat{p}^{S_2}(1-\hat{p})^{n-S_2}} \\
& = n \left(\hat{p}_1 \log \frac{\hat{p}_1}{\hat{p}} + (1-\hat{p}_1) \log \frac{1-\hat{p}_1}{1-\hat{p}}\right) + n \left(\hat{p}_2 \log \frac{\hat{p}_2}{\hat{p}} + (1-\hat{p}_2) \log \frac{1-\hat{p}_2}{1-\hat{p}}\right) \\
& = n \mathrm{kl}(\hat{p}_1,\hat{p}) + n \mathrm{kl}(\hat{p}_2,\hat{p})
\end{align}

where we introduce the KL divergence between two Bernoulli distributions with mean $p$ and $q$ as $\mathrm{kl}(p,q) = p\log\frac{p}{q} + (1-p)\log \frac{1-p}{1-q}$

For unbalanced sample sizes $n_1$ and $n_2$ we get 
$$\log \widetilde{\Lambda}(X,Y) = n_1\mathrm{kl}(\hat{p}_{1,n_1},\hat{p}) + n_2 \mathrm{kl}(\hat{p}_{2,n_2},\hat{p})$$
with the pooled estimator being $\widehat{p} = \frac{n_1}{n_1+n_2}\hat{p}_{1,n_1} + \frac{n_2}{n_1+n_2}\hat{p}_{2,n_2}$. 

**Complete the code below to compute the log-likelihood ratio.**

In [ ]:
eps = 1e-15    

def kl(p, q):
    """Kullback-Leibler divergence for Bernoulli distributions."""
    p = min(max(p, eps), 1-eps)
    q = min(max(q, eps), 1-eps)
    return p*np.log(p/q) + (1-p)*np.log((1-p)/(1-q))

def Log_Likelihood_Ratio(X,Y): 
    pass

A LRT test is of the form 
$$D(X,Y) = \mathbb{1}\left(\log\widetilde{\Lambda}(X,Y) > t\right)$$
for some threshold $t$. 

**Complete the function below to implement a LRT guaranteed to have an asymptotic type one error of $\alpha$ using Wilk's theorem.**

In [ ]:
def LRT_test(X,Y,alpha=0.05):
    test_statistic = Log_Likelihood_Ratio(X,Y) 
    threshold = ## TO COMPLETE 
    decision = 0
    if (test_statistic>threshold):
        decision = 1
    return decision 

## Running the test 

As before, you can run the test with varying parameters, and also compare its outcome to the Wald test.

In [ ]:
n = 453 # number of patients in each group
p1 = 0.8
p2 = 0.85

rng = np.random.RandomState(42)
X = (rng.random(n)<p1).astype(int)
print("average efficacy of treatment 1 is", np.mean(X))
Y = (rng.random(n)<p2).astype(int)
print("average efficacy of treatment 2 is", np.mean(Y))

print("decision of the LRT test:", LRT_test(X,Y))
print("decision of the Wald test:", Wald_test(X,Y))

## Estimation of the test characteristics

Below, we estimate empirically the type I error of the two tests using simulations. 

In [ ]:
p=0.95 # common value of p1 and p2 for an element in H0
alpha_0 = 0.05 # desired type I error 

M = 100 # number of simulations 
n_values = np.array([30,50,100,200,300,500]) # number of patients   
n_max =np.max(n_values)
n_nb = len(n_values)

X_table = (np.random.random(size=(M,n_max))<p).astype(int)
Y_table = (np.random.random(size=(M,n_max))<p).astype(int)

alphas_LRT = np.zeros(n_nb)
alphas_Wald = np.zeros(n_nb)

for i in range(n_nb):
    n = n_values[i] 
    for m in range(M):
        X_small = X_table[m,range(n)]
        Y_small = Y_table[m,range(n)]
        decision=Wald_test(X_small,Y_small,alpha_0)
        alphas_Wald[i]=alphas_Wald[i]+decision
        decision=LRT_test(X_small,Y_small,alpha_0)
        alphas_LRT[i]=alphas_LRT[i]+decision
    alphas_Wald[i]=alphas_Wald[i]/M
    alphas_LRT[i]=alphas_LRT[i]/M

plt.plot(n_values,alphas_Wald,label="Wald test")
plt.plot(n_values,alphas_LRT,label="LRT test")
plt.axhline(alpha_0,0, 1, c='black', linewidth = 1)
plt.title("empirical type I error at p1=p2={} as a function of the sample size".format(p))
plt.legend()
plt.show()


**And you can do the same for the power.** 

**Explore different values of the parameters p, p1, p2 to investigate potential differences in behavior between the two tests.**

# Wald Test: Pooled versus UnPooled variance

Recalling that $\widehat{p}_n = \frac{1}{2n}\sum_{i=1}^{n}(X_i+Y_i)$ is the estimator of the probability of sucess assuming the two populations come from the same distribution, we can define the test statistic

$$W_n^{(p)} = \frac{\hat{p}_{1,n} - \hat{p}_{2,n}}{\sqrt{\frac{\hat{p}_{n}(1-\hat{p}_{n})}{n} + \frac{ \hat{p}_{n}(1- \hat{p}_{n})}{n}}}$$ 

The difference with $W_n$ is that the variance estimator is computed assuming all a similar variance for both populations: this is called the pooled variance.

**Justify that $W_n^{(p)}$ is also asymptotically normal under H_0. Compare the resulting test to that based on $W_n$.**